In [ ]:
pip install pyalex


In [ ]:
import time
import requests
import pandas as pd
import math
from tqdm import tqdm

# Configuration
headers = {"User-Agent": "mailto:your_student_email@vinuni.edu.vn"}
BASE_URL = "https://api.openalex.org/works"

START_YEAR = 2015
END_YEAR = 2024
SAMPLING_RATE = 0.02
# Using a keyword search instead of a strict concept ID to ensure we find results
SEARCH_QUERY = "artificial intelligence"

def flatten_work(work):
    """Extract and flatten relevant fields from an OpenAlex work object."""
    institutions = set()
    countries = set()

    for authorship in work.get("authorships", []):
        for inst in authorship.get("institutions", []):
            if inst.get("display_name"):
                institutions.add(inst.get("display_name"))
            if inst.get("country_code"):
                countries.add(inst.get("country_code"))

    primary_loc = work.get("primary_location") or {}
    source = primary_loc.get("source") or {}
    venue = source.get("display_name", "Unknown Venue")

    return {
        "openalex_id": work.get("id"),
        "title": work.get("display_name"),
        "publication_year": int(work.get("publication_year")),
        "citation_count": work.get("cited_by_count", 0),
        "venue": venue,
        "institutions": "; ".join(sorted(institutions)),
        "countries": "; ".join(sorted(countries)),
        "primary_country": list(countries)[0] if countries else None
    }

def crawl_proportional():
    all_clean_data = []

    for year in range(START_YEAR, END_YEAR + 1):
        # Filter string using search instead of concept to be more inclusive
        filter_str = f"publication_year:{year},has_institutions:true,type:article|conference-proceeding"
        query_url = f"{BASE_URL}?search={SEARCH_QUERY}&filter={filter_str}"

        # Step 1: Get total paper count for the year
        try:
            res_meta = requests.get(f"{query_url}&per_page=1", headers=headers).json()
            total_papers_real = res_meta.get("meta", {}).get("count", 0)
        except Exception as e:
            print(f"Error fetching count for {year}: {e}")
            continue

        if total_papers_real == 0:
            print(f"No papers found for {year} with query '{SEARCH_QUERY}'")
            continue

        # Step 2: Calculate target sample size
        papers_to_sample = math.ceil(total_papers_real * SAMPLING_RATE)
        print(f"\n--- Year {year} --- Total: {total_papers_real:,} | Target (2%): {papers_to_sample:,}")

        papers_collected_this_year = 0
        cursor = "*"
        pbar = tqdm(total=papers_to_sample, desc=f"Fetching {year}")

        while papers_collected_this_year < papers_to_sample:
            current_per_page = min(200, papers_to_sample - papers_collected_this_year)

            url = (
                f"{query_url}&"
                f"sample={papers_to_sample}&"
                f"select=id,display_name,publication_year,cited_by_count,authorships,primary_location&"
                f"per_page={current_per_page}&cursor={cursor}"
            )

            try:
                response = requests.get(url, headers=headers, timeout=30)
                data = response.json()
                results = data.get("results", [])

                if not results:
                    break

                for work in results:
                    flattened = flatten_work(work)
                    if flattened["primary_country"]:
                        all_clean_data.append(flattened)
                        papers_collected_this_year += 1
                        pbar.update(1)

                cursor = data.get("meta", {}).get("next_cursor")
                if not cursor:
                    break

                time.sleep(0.1)
            except Exception as e:
                print(f"Error in loop: {e}")
                time.sleep(2)
                continue

        pbar.close()

    if not all_clean_data:
        print("No data collected. Check filters or search terms.")
        return pd.DataFrame()

    df = pd.DataFrame(all_clean_data)
    df.to_parquet("proportional_ai_trends.parquet", index=False)

    print("\n--- Collection Summary ---")
    print(df["publication_year"].value_counts().sort_index())
    print(f"Total collected: {len(df):,}")
    return df

if __name__ == "__main__":
    df_final = crawl_proportional()
    if not df_final.empty:
        display(df_final.head())

No papers found for 2015 with query 'artificial intelligence'
No papers found for 2016 with query 'artificial intelligence'
No papers found for 2017 with query 'artificial intelligence'
No papers found for 2018 with query 'artificial intelligence'
No papers found for 2019 with query 'artificial intelligence'
No papers found for 2020 with query 'artificial intelligence'
No papers found for 2021 with query 'artificial intelligence'
No papers found for 2022 with query 'artificial intelligence'
No papers found for 2023 with query 'artificial intelligence'
No papers found for 2024 with query 'artificial intelligence'
No data collected. Check filters or search terms.


In [ ]:
import time
import requests
import pandas as pd
import math
from tqdm import tqdm

# Configuration
headers = {"User-Agent": "mailto:your_student_email@vinuni.edu.vn"}
BASE_URL = "https://api.openalex.org/works"

START_YEAR = 2015
END_YEAR = 2024
SAMPLING_RATE = 0.02
SEARCH_QUERY = "artificial intelligence"

def flatten_work(work):
    """Extract and flatten relevant fields from an OpenAlex work object."""
    institutions = set()
    countries = set()

    for authorship in work.get("authorships", []):
        for inst in authorship.get("institutions", []):
            if inst.get("display_name"):
                institutions.add(inst.get("display_name"))
            if inst.get("country_code"):
                countries.add(inst.get("country_code"))

    primary_loc = work.get("primary_location") or {}
    source = primary_loc.get("source") or {}
    venue = source.get("display_name", "Unknown Venue")

    return {
        "openalex_id": work.get("id"),
        "title": work.get("display_name"),
        "publication_year": int(work.get("publication_year")),
        "citation_count": work.get("cited_by_count", 0),
        "venue": venue,
        "institutions": "; ".join(sorted(institutions)),
        "countries": "; ".join(sorted(countries)),
        "primary_country": list(countries)[0] if countries else None
    }

def crawl_proportional():
    all_clean_data = []

    for year in range(START_YEAR, END_YEAR + 1):
        # Standard way to combine keyword search (q) and filters
        filter_str = f"publication_year:{year},has_institutions:true,type:article"
        query_params = f"q={SEARCH_QUERY}&filter={filter_str}"

        try:
            res_meta = requests.get(f"{BASE_URL}?{query_params}&per_page=1", headers=headers).json()
            total_papers_real = res_meta.get("meta", {}).get("count", 0)
        except Exception as e:
            print(f"Error fetching count for {year}: {e}")
            continue

        if total_papers_real == 0:
            print(f"No papers found for {year} with query '{SEARCH_QUERY}'")
            continue

        papers_to_sample = math.ceil(total_papers_real * SAMPLING_RATE)
        # Safe limit for a single sample request
        papers_to_sample = min(papers_to_sample, 5000)
        print(f"\n--- Year {year} --- Total Real: {total_papers_real:,} | Target Sample (2%): {papers_to_sample:,}")

        # Retrieve the random sample using 'sample' parameter
        sample_url = (
            f"{BASE_URL}?{query_params}&"
            f"sample={papers_to_sample}&"
            f"select=id,display_name,publication_year,cited_by_count,authorships,primary_location&"
            f"per_page=200"
        )

        try:
            response = requests.get(sample_url, headers=headers, timeout=30).json()
            results = response.get("results", [])

            pbar = tqdm(total=len(results), desc=f"Processing {year}")
            for work in results:
                flattened = flatten_work(work)
                if flattened["primary_country"]:
                    all_clean_data.append(flattened)
                pbar.update(1)
            pbar.close()
        except Exception as e:
            print(f"Error fetching results for {year}: {e}")
            continue

        time.sleep(0.5)

    if not all_clean_data:
        print("No data collected. Try adjusting the search query or filters.")
        return pd.DataFrame()

    df = pd.DataFrame(all_clean_data)
    df.to_parquet("proportional_ai_trends.parquet", index=False)
    df.to_csv("proportional_ai_trends.csv", index=False)
    print("\n--- Collection Done ---")
    print(df["publication_year"].value_counts().sort_index())
    return df

if __name__ == "__main__":
    df_final = crawl_proportional()
    if not df_final.empty:
        display(df_final.head())

No papers found for 2015 with query 'artificial intelligence'
No papers found for 2016 with query 'artificial intelligence'
No papers found for 2017 with query 'artificial intelligence'
No papers found for 2018 with query 'artificial intelligence'
No papers found for 2019 with query 'artificial intelligence'
No papers found for 2020 with query 'artificial intelligence'
No papers found for 2021 with query 'artificial intelligence'
No papers found for 2022 with query 'artificial intelligence'
No papers found for 2023 with query 'artificial intelligence'
No papers found for 2024 with query 'artificial intelligence'
No data collected. Try adjusting the search query or filters.


In [ ]:
from google.colab import files
files.download('openalex_ai_papers.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# 0. INSTALL / IMPORT
# ============================================================

import os
import json
import time
import math
import getpass
from pathlib import Path

import requests
import pandas as pd
import numpy as np


# ============================================================
# 1. CONFIG
# ============================================================

# In Colab, recommended:
# Runtime > Secrets > Add secret named OPENALEX_API_KEY
# Or just paste it securely when asked below.

try:
    from google.colab import userdata
    OPENALEX_API_KEY = userdata.get("OPENALEX_API_KEY")
except Exception:
    OPENALEX_API_KEY = None

if not OPENALEX_API_KEY:
    OPENALEX_API_KEY = getpass.getpass("Paste your OpenAlex API key: ")

EMAIL = "your.email@example.com"

START_YEAR = 2000
END_YEAR = 2026
TARGET_SAMPLE_SIZE = 30000

OUTPUT_DIR = Path("openalex_ai_30k")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_JSON_PATH = OUTPUT_DIR / "ai_30k_raw.json"
CLEAN_CSV_PATH = OUTPUT_DIR / "ai_30k_clean.csv"
YEAR_COUNTS_PATH = OUTPUT_DIR / "ai_exact_year_counts.csv"
TOPIC_LONG_PATH = OUTPUT_DIR / "ai_topics_long.csv"
COUNTRY_LONG_PATH = OUTPUT_DIR / "ai_countries_long.csv"
INSTITUTION_LONG_PATH = OUTPUT_DIR / "ai_institutions_long.csv"

BASE_URL = "https://api.openalex.org/works"

# Main definition of AI papers:
# primary_topic.subfield.id:1702 = Artificial Intelligence subfield in OpenAlex
BASE_FILTER = ",".join([
    "primary_topic.subfield.id:1702",
    "has_doi:true",
    "is_retracted:false",
    "is_paratext:false",
    "type:article|preprint|book-chapter|proceedings-article",
    f"from_publication_date:{START_YEAR}-01-01",
    f"to_publication_date:{END_YEAR}-12-31",
])

# Keep only fields needed for your visualization project.
SELECT_FIELDS = ",".join([
    "id",
    "doi",
    "title",
    "display_name",
    "publication_year",
    "publication_date",
    "type",
    "cited_by_count",
    "referenced_works_count",
    "authorships",
    "topics",
    "primary_topic",
    "primary_location",
    "language",
    "open_access",
    "is_retracted",
    "is_paratext",
])


# ============================================================
# 2. REQUEST HELPER
# ============================================================

def request_openalex(params, max_retries=6, sleep_base=2):
    """
    Request OpenAlex using api_key query parameter.
    Retries on temporary errors or rate limits.
    """
    params = dict(params)
    params["api_key"] = OPENALEX_API_KEY
    params["mailto"] = EMAIL

    headers = {
        "User-Agent": f"mailto:{EMAIL}"
    }

    for attempt in range(max_retries):
        response = requests.get(
            BASE_URL,
            params=params,
            headers=headers,
            timeout=90
        )

        if response.status_code == 200:
            return response.json()

        wait = sleep_base ** attempt
        print(f"OpenAlex error {response.status_code}. Waiting {wait}s. Response: {response.text[:300]}")
        time.sleep(wait)

    raise RuntimeError("OpenAlex request failed after retries.")


def safe_get(d, keys, default=None):
    cur = d
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
        if cur is None:
            return default
    return cur


def safe_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def join_unique(values):
    values = safe_list(values)
    cleaned = [str(v) for v in values if v not in [None, "", "nan"]]
    return "; ".join(sorted(set(cleaned)))


# ============================================================
# 3. GET EXACT YEARLY COUNTS
# ============================================================

print("Getting exact yearly counts from OpenAlex...")

year_count_data = request_openalex({
    "filter": BASE_FILTER,
    "group_by": "publication_year",
    "per-page": 200,
})

groups = year_count_data.get("group_by", [])

year_rows = []
for g in groups:
    year = int(g["key"])
    if START_YEAR <= year <= END_YEAR:
        year_rows.append({
            "publication_year": year,
            "exact_ai_paper_count": int(g["count"])
        })

year_counts = pd.DataFrame(year_rows)

# Ensure every year exists, even if count is zero.
all_years = pd.DataFrame({"publication_year": list(range(START_YEAR, END_YEAR + 1))})
year_counts = all_years.merge(year_counts, on="publication_year", how="left")
year_counts["exact_ai_paper_count"] = year_counts["exact_ai_paper_count"].fillna(0).astype(int)

year_counts = year_counts.sort_values("publication_year").reset_index(drop=True)

total_population = year_counts["exact_ai_paper_count"].sum()

print(year_counts)
print("\nTotal OpenAlex AI-paper population under this filter:", total_population)

if total_population == 0:
    raise ValueError("No papers found. Check your API key or filter syntax.")


# ============================================================
# 4. ALLOCATE 30K STRATIFIED SAMPLE BY YEAR
# ============================================================

year_counts["target_n"] = (
    year_counts["exact_ai_paper_count"] / total_population * TARGET_SAMPLE_SIZE
).round().astype(int)

# Make sure non-empty years get at least 1 paper.
year_counts.loc[
    (year_counts["exact_ai_paper_count"] > 0) & (year_counts["target_n"] == 0),
    "target_n"
] = 1

# Adjust rounding to exactly 30k.
diff = TARGET_SAMPLE_SIZE - year_counts["target_n"].sum()

if diff != 0:
    idxs = year_counts.sort_values("exact_ai_paper_count", ascending=False).index.tolist()
    step = 1 if diff > 0 else -1

    moved = 0
    i = 0

    while moved < abs(diff):
        idx = idxs[i % len(idxs)]

        if year_counts.loc[idx, "target_n"] + step >= 0:
            year_counts.loc[idx, "target_n"] += step
            moved += 1

        i += 1

year_counts["sample_weight"] = np.where(
    year_counts["target_n"] > 0,
    year_counts["exact_ai_paper_count"] / year_counts["target_n"],
    np.nan
)

year_counts.to_csv(YEAR_COUNTS_PATH, index=False, encoding="utf-8-sig")

print("\nSampling plan:")
print(year_counts)
print("\nTotal planned sample:", year_counts["target_n"].sum())


# ============================================================
# 5. FETCH RANDOM SAMPLE FOR ONE YEAR
# ============================================================

def fetch_sample_for_year(year, n, sample_weight, seed_base=20260601):
    """
    Fetch n random works from one year using OpenAlex sample + page pagination.
    This fixes the 200-per-year limit.
    """
    if n <= 0:
        return []

    year_filter = BASE_FILTER + f",publication_year:{year}"

    collected = []
    seen = set()

    seed = seed_base + int(year)
    total_pages = math.ceil(n / 200)

    for page_num in range(1, total_pages + 1):
        remaining = n - len(collected)
        per_page = min(200, remaining)

        if remaining <= 0:
            break

        data = request_openalex({
            "filter": year_filter,
            "sample": n,
            "seed": seed,
            "page": page_num,
            "per-page": per_page,
            "select": SELECT_FIELDS,
        })

        batch = data.get("results", [])

        if not batch:
            print(f"{year}: empty batch at page {page_num}")
            break

        for paper in batch:
            pid = paper.get("id")

            if pid and pid not in seen:
                paper["_sample_year"] = year
                paper["_sample_weight"] = float(sample_weight)
                collected.append(paper)
                seen.add(pid)

            if len(collected) >= n:
                break

        print(f"{year}: page {page_num}/{total_pages}, collected {len(collected)} / {n}")

        time.sleep(0.15)

    return collected


# ============================================================
# 6. CRAWL 30K STRATIFIED SAMPLE
# ============================================================

print("\nStarting stratified random sample crawl...")

all_papers = []
global_seen = set()

for _, row in year_counts.iterrows():
    year = int(row["publication_year"])
    n = int(row["target_n"])
    sample_weight = row["sample_weight"]

    if n <= 0:
        continue

    papers_year = fetch_sample_for_year(year, n, sample_weight)

    unique_year_papers = []

    for paper in papers_year:
        pid = paper.get("id")
        if pid and pid not in global_seen:
            unique_year_papers.append(paper)
            global_seen.add(pid)

    all_papers.extend(unique_year_papers)

    print(f"{year}: collected {len(unique_year_papers)} / {n}")

print(f"\nTotal collected unique papers: {len(all_papers)}")


# ============================================================
# 7. SAVE RAW JSON
# ============================================================

with open(RAW_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(all_papers, f, ensure_ascii=False, indent=2)

print(f"Saved raw JSON to: {RAW_JSON_PATH}")


# ============================================================
# 8. FLATTEN TO CLEAN CSV
# ============================================================

rows = []
topic_long_rows = []
country_long_rows = []
institution_long_rows = []

for paper in all_papers:
    paper_id = paper.get("id")
    publication_year = paper.get("publication_year")
    citation_count = paper.get("cited_by_count") or 0

    if publication_year:
        paper_age = max(1, END_YEAR - int(publication_year) + 1)
        citations_per_year = citation_count / paper_age
    else:
        paper_age = np.nan
        citations_per_year = np.nan

    # ----------------------------
    # Authors / institutions / countries
    # ----------------------------
    authors = []
    institutions = []
    institution_ids = []
    countries = []

    for auth in safe_list(paper.get("authorships")):
        author_name = safe_get(auth, ["author", "display_name"])
        if author_name:
            authors.append(author_name)

        for c in safe_list(auth.get("countries")):
            if c:
                countries.append(c)

        for inst in safe_list(auth.get("institutions")):
            inst_id = inst.get("id")
            inst_name = inst.get("display_name")
            inst_country = inst.get("country_code")
            inst_type = inst.get("type")

            if inst_name:
                institutions.append(inst_name)

            if inst_id:
                institution_ids.append(inst_id)

            if inst_country:
                countries.append(inst_country)

            if inst_name:
                institution_long_rows.append({
                    "paper_id": paper_id,
                    "publication_year": publication_year,
                    "institution_id": inst_id,
                    "institution": inst_name,
                    "institution_country": inst_country,
                    "institution_type": inst_type,
                    "citation_count": citation_count,
                    "sample_weight": paper.get("_sample_weight"),
                })

    # ----------------------------
    # Topics
    # ----------------------------
    topics = []

    for topic in safe_list(paper.get("topics")):
        topic_name = topic.get("display_name")
        topic_id = topic.get("id")
        topic_score = topic.get("score")

        subfield = safe_get(topic, ["subfield", "display_name"])
        field = safe_get(topic, ["field", "display_name"])
        domain = safe_get(topic, ["domain", "display_name"])

        if topic_name:
            topics.append(topic_name)

            topic_long_rows.append({
                "paper_id": paper_id,
                "publication_year": publication_year,
                "topic_id": topic_id,
                "topic": topic_name,
                "topic_score": topic_score,
                "subfield": subfield,
                "field": field,
                "domain": domain,
                "citation_count": citation_count,
                "sample_weight": paper.get("_sample_weight"),
            })

    # ----------------------------
    # Countries long
    # ----------------------------
    countries_unique = sorted(set([c for c in countries if c]))

    for c in countries_unique:
        country_long_rows.append({
            "paper_id": paper_id,
            "publication_year": publication_year,
            "country": c,
            "citation_count": citation_count,
            "sample_weight": paper.get("_sample_weight"),
            "fractional_paper_weight": paper.get("_sample_weight") / len(countries_unique)
            if len(countries_unique) > 0 else np.nan,
        })

    # ----------------------------
    # Main paper row
    # ----------------------------
    rows.append({
        # Publication metadata
        "paper_id": paper_id,
        "title": paper.get("title") or paper.get("display_name"),
        "publication_year": publication_year,
        "publication_date": paper.get("publication_date"),
        "publication_type": paper.get("type"),

        # Impact metrics
        "citation_count": citation_count,
        "citations_per_year": citations_per_year,
        "referenced_works_count": paper.get("referenced_works_count"),

        # Research context
        "topics": join_unique(topics),
        "primary_topic": safe_get(paper, ["primary_topic", "display_name"]),
        "primary_subfield": safe_get(paper, ["primary_topic", "subfield", "display_name"]),
        "primary_field": safe_get(paper, ["primary_topic", "field", "display_name"]),
        "primary_domain": safe_get(paper, ["primary_topic", "domain", "display_name"]),
        "venue_source": safe_get(paper, ["primary_location", "source", "display_name"]),
        "venue_type": safe_get(paper, ["primary_location", "source", "type"]),

        # Contributor metadata
        "authors": join_unique(authors),
        "institutions": join_unique(institutions),
        "institution_ids": join_unique(institution_ids),
        "countries": join_unique(countries_unique),

        # Extra useful metadata
        "doi": paper.get("doi"),
        "language": paper.get("language"),
        "is_oa": safe_get(paper, ["open_access", "is_oa"]),
        "oa_status": safe_get(paper, ["open_access", "oa_status"]),

        # Sampling metadata
        "sample_year": paper.get("_sample_year"),
        "sample_weight": paper.get("_sample_weight"),
    })

df = pd.DataFrame(rows).drop_duplicates(subset=["paper_id"])
topic_long = pd.DataFrame(topic_long_rows).drop_duplicates()
country_long = pd.DataFrame(country_long_rows).drop_duplicates()
institution_long = pd.DataFrame(institution_long_rows).drop_duplicates()

df.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8-sig")
topic_long.to_csv(TOPIC_LONG_PATH, index=False, encoding="utf-8-sig")
country_long.to_csv(COUNTRY_LONG_PATH, index=False, encoding="utf-8-sig")
institution_long.to_csv(INSTITUTION_LONG_PATH, index=False, encoding="utf-8-sig")

print("\nSaved files:")
print(CLEAN_CSV_PATH)
print(YEAR_COUNTS_PATH)
print(TOPIC_LONG_PATH)
print(COUNTRY_LONG_PATH)
print(INSTITUTION_LONG_PATH)

print("\nMain CSV shape:", df.shape)
print(df.head())

Paste your OpenAlex API key: ··········
Getting exact yearly counts from OpenAlex...
    publication_year  exact_ai_paper_count
0               2000                 21708
1               2001                 20773
2               2002                 46553
3               2003                 35108
4               2004                 33854
5               2005                 41205
6               2006                 45342
7               2007                 45229
8               2008                 47871
9               2009                 52189
10              2010                 54957
11              2011                 56938
12              2012                 60576
13              2013                 63323
14              2014                 69592
15              2015                 68515
16              2016                 76714
17              2017                 84560
18              2018                 99466
19              2019                113703
20          

In [ ]:
from google.colab import files

# Trigger download for the clean CSV file
files.download('openalex_ai_30k/ai_30k_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>